# Clase 10 - Atención y arquitectura Transformer

Procesamiento de Lenguaje Natural - Felipe Castillo Rendón

## D0 - Preparemos el entorno sin cargar todo el curso

Hoy las entradas son matrices pequeñas escritas dentro del notebook. **No necesitas CSV, ZIP, Drive, claves de acceso ni descargar un modelo.** Vamos a usar CPU, el procesador de la computadora.

En Colab, abre este archivo y ejecuta las celdas en orden. En Jupyter local, selecciona el entorno del curso con Python 3.11 y PyTorch. Si falta una biblioteca, instala `numpy pandas torch` en el entorno del notebook y reinicia el kernel, es decir, el proceso que ejecuta Python. No reinstales todo si los imports ya funcionan.

**Entrada:** las bibliotecas instaladas. **Salida:** versiones y dispositivo; aún no hemos calculado atención. La semilla 42 permite repetir el inicio aleatorio en el mismo entorno; no garantiza igualdad exacta entre todas las versiones.

Antes de ejecutar: ¿necesitamos entrenar un modelo para comprobar una multiplicación de matrices?

> No necesariamente, puede utilizarse, pero no es un must. Para comprobar la multiplicación de matrices y el softmax basta con realizar las operaciones matemáticas, no hay pesos que aprender. Si usamos un modelo entrenado, ya no estaríamos verificando la operación, sino viendo qué hacen unos pesos ya aprendidos.


In [1]:
import math
import platform
import numpy as np
import pandas as pd
import torch
from IPython.display import display

# Trabajamos en CPU y repetimos el mismo inicio aleatorio.
torch.manual_seed(42)
torch.set_num_threads(2)
torch.set_printoptions(precision=6, sci_mode=False)
print({'Python': platform.python_version(), 'NumPy': np.__version__,
       'pandas': pd.__version__, 'PyTorch': torch.__version__, 'dispositivo': 'cpu'})

{'Python': '3.13.18', 'NumPy': '2.4.6', 'pandas': '2.2.3', 'PyTorch': '2.11.0+cpu', 'dispositivo': 'cpu'}


### Leamos lo que ocurrió

Si aparece un error de importación, todavía no es un error del cálculo: falta preparar el entorno. Si ves las versiones, continúa. Esta clase no utiliza corpus ni particiones train/test; inspeccionaremos un mecanismo con números elegidos y un bloque sin entrenar.

## D1 - ¿Qué recibe la atención? Sigamos Q, K y V

Imagina una posición que necesita combinar información de otras dos. **Q** es la consulta (*query*), **K** contiene las claves (*keys*) con las que la comparamos y **V** contiene los valores (*values*) que mezclaremos. Q no es una pregunta en español: es una lista de números.

Una **matriz** organiza números en filas y columnas. Un **tensor** es la estructura que usa PyTorch para estas matrices y para arreglos con más ejes. Aquí Q tiene una fila y dos columnas; cada fila de K y V corresponde a la misma posición.

Recibimos `Q=[[1,0]]`, `K=[[1,0],[0,1]]` y `V=[[2,0],[0,4]]`. Son números didácticos, no representaciones aprendidas de palabras. Con `@` hacemos un producto matricial: multiplicamos coordenadas correspondientes y sumamos. `K.T` intercambia filas y columnas; en este K particular no cambia sus valores porque es la matriz identidad.

Calculamos 1×1+0×0=1 y 1×0+0×1=0. Después dividimos entre √2≈1.414214. La raíz cuadrada de 2 es el número que multiplicado por sí mismo da 2. `math.sqrt(2)` la calcula.

Antes de ejecutar: ¿qué clave tendrá mayor puntuación? La salida separará producto y puntuación escalada; todavía no son pesos de mezcla.

> La primera clave, ya que la consulta Q coincide con K₁ en la coordenada donde Q tiene 1, por lo que el producto da 1. Con K₂= el producto da 0.

In [2]:
Q = torch.tensor([[1., 0.]])
K = torch.tensor([[1., 0.], [0., 1.]])
V = torch.tensor([[2., 0.], [0., 4.]])

dk = Q.shape[-1]  # Dos coordenadas por consulta y por clave.
product = Q @ K.T
scores = product / math.sqrt(dk)
print('Formas Q, K, V:', tuple(Q.shape), tuple(K.shape), tuple(V.shape))
display(pd.DataFrame({'clave': ['posición 1', 'posición 2'],
                      'producto': product[0].numpy(),
                      'score_escalado': scores[0].numpy()}))
assert product.shape == (1, 2)

Formas Q, K, V: (1, 2) (2, 2) (2, 2)


,clave,producto,score_escalado
0,posición 1,1.0,0.707107
1,posición 2,0.0,0.000000


### Leamos lo que ocurrió

La fila de Q compara una consulta con dos claves: obtenemos una fila con dos puntuaciones. `score_escalado` significa puntuación después de dividir entre √dk. Esa escala modera la magnitud, pero no hace que las puntuaciones sumen 1. En general, si Q es Lq×dk y K es Lk×dk, QKᵀ es Lq×Lk: una fila por consulta y una columna por clave.

## D2 - Convirtamos las puntuaciones en pesos y después en una mezcla

Recibimos `scores` y V de D1. **Softmax** convierte una fila de puntuaciones en pesos positivos cuya suma es 1. Primero calcula una exponencial y luego divide cada resultado entre el total de esa fila.

`exp(x)=e^x`, con e≈2.71828; `^` representa una potencia en esta explicación, pero en Python escribimos `math.exp(x)` o `torch.exp(x)`. Es la operación inversa del logaritmo natural `ln`: si ln(y)=x, entonces exp(x)=y.

Cuenta directa: exp(0.70710678)≈2.028115 y exp(0)=1. Total≈3.028115. Pesos: 2.028115/3.028115≈0.669762 y 1/3.028115≈0.330238.

El código también resta el máximo de la fila antes de exp. Así evita números demasiado grandes: exp(0)=1 y exp(−0.70710678)≈0.493069. **Cambian las exponenciales, pero no los pesos**: dividimos numerador y denominador por el mismo factor.

Por último combinamos V: [0.669762×2+0.330238×0, 0.669762×0+0.330238×4]. Antes de ejecutar: ¿la salida tendrá dos números o dos etiquetas?

> La salida tendrá dos números, porque weights @ V hereda el número de columnas de V, no sus filas. V es 2×2 aquí, así que salen dos coordenadas que son números y no etiquetas dado que se mezclan las filas de V con los pesos, obteniendo como resultado valores numéricos, no clasificaciones en sí mismas.

In [3]:
direct_exp = torch.exp(scores)
stable_exp = torch.exp(scores - scores.max(dim=-1, keepdim=True).values)
weights = stable_exp / stable_exp.sum(dim=-1, keepdim=True)
out = weights @ V

display(pd.DataFrame({'clave': ['posición 1', 'posición 2'],
    'exp_directa': direct_exp[0].numpy(),
    'exp_estable': stable_exp[0].numpy(), 'peso': weights[0].numpy()}))
print('Suma de pesos:', weights.sum(dim=-1))
print('Salida de la mezcla:', out)
# Comprobamos el desarrollo manual con la operación de PyTorch.
assert torch.allclose(weights, torch.softmax(scores, dim=-1), atol=1e-6, rtol=0)
assert torch.allclose(weights.sum(dim=-1), torch.ones(1), atol=1e-6, rtol=0)

,clave,exp_directa,exp_estable,peso
0,posición 1,2.028115,1.000000,0.669762
1,posición 2,1.000000,0.493069,0.330238


Suma de pesos: tensor([1.])
Salida de la mezcla: tensor([[1.339523, 1.320954]])


### Leamos lo que ocurrió

Cada fila reparte peso entre sus claves. `dim=-1` elige el último eje, las columnas; `keepdim=True` conserva ese eje para dividir la fila correctamente. La salida es un vector de dos coordenadas: no es una categoría ni una frase. El peso 0.669762 indica cuánto participa el primer valor en esta mezcla; no significa 66.98 % de certeza de que una afirmación sea verdadera.

Después: reconstruye una coordenada de la salida y explica por qué las dos columnas de exponenciales producen los mismos pesos.

> Para la primera coordenada de la salida: 0.669762×2 + 0.330238×0 = 1.339524 aproximadamente. Las dos columnas de exponenciales dan los mismos pesos porque lo único que cambia entre exp_directa y exp_estable es que a la segunda le restamos el máximo de la fila antes de exponenciar. Restar una constante c a todas las puntuaciones multiplica el numerador y el denominador por exp(-c), que se cancela en la división.


## D3 - Bloqueemos una clave antes de normalizar

Una **máscara** indica qué posiciones pueden participar. Recibimos las puntuaciones de D1 y el mismo V. Queremos excluir la segunda clave, sin cambiar la primera ni los valores.

Colocamos −∞ (menos infinito) en la puntuación prohibida **antes** de softmax. exp(−∞) se interpreta como el límite 0. En consecuencia, esa posición no recibe peso. No significa que el dato original se haya borrado.

Antes de ejecutar, anota qué esperas en los dos pesos y en la salida. También veremos un error deliberado: poner un peso a cero después de normalizar sin volver a repartir el total.

> Supongo que los dos pesos deberían quedar entre 0 y 1. Al bloquear la segunda clave con infinito negativo antes de softmax, exp(-inf) debe de dar como resultado 0 y la primera se lleva todo el peso, por lo que quedaría [1, 0] por los rangos en los que se pueden mover los resultados. La salida sería [1,0] @ V = [2, 0].


In [4]:
blocked = scores.clone()  # Conservamos scores para poder comparar.
blocked[:, 1] = float('-inf')
masked_weights = torch.softmax(blocked, dim=-1)
masked_out = masked_weights @ V
print('Scores con exclusión:', blocked)
print('Pesos correctos:', masked_weights, 'Salida:', masked_out)

wrong_weights = weights.clone()
wrong_weights[:, 1] = 0  # Error deliberado: ahora la fila ya no suma 1.
print('Contraste incorrecto: suma =', wrong_weights.sum().item(),
      'mezcla =', wrong_weights @ V)
assert torch.allclose(masked_weights, torch.tensor([[1., 0.]]), atol=1e-6, rtol=0)
assert torch.allclose(masked_out, torch.tensor([[2., 0.]]), atol=1e-6, rtol=0)

Scores con exclusión: tensor([[0.707107,     -inf]])
Pesos correctos: tensor([[1., 0.]]) Salida: tensor([[2., 0.]])
Contraste incorrecto: suma = 0.6697615385055542 mezcla = tensor([[1.339523, 0.000000]])


### Leamos lo que ocurrió

La única clave permitida recibe todo el peso. Bloquear después de softmax sin volver a normalizar reduce la suma y cambia la mezcla. No bloquees todas las claves de una consulta: no queda una distribución válida y pueden aparecer valores NaN, es decir, un resultado numérico no definido.

Después: explica por qué permitir una sola clave no obliga a que la salida sea [1,0]. El peso y el valor tienen funciones distintas.

> Permitir una sola clave no obliga a que la salida sea [1,0] porque el peso y el valor hacen cosas distintas. El peso dice cuánto participa cada clave (en [1, 0] la primera se lleva todo). La salida es la mezcla ponderada de los values.

## D4 - Tres posiciones: nadie puede consultar el futuro

Recibimos Q=K=V=I de tamaño 3×3. I es la identidad: unos en la diagonal y ceros fuera. Cada fila representa una posición distinta.

Una máscara **causal** deja a cada posición consultar su pasado y a sí misma. Así, al predecir el siguiente token, no puede usar información futura. La fila 1 permite la columna 1; la fila 2, las columnas 1 y 2; la fila 3, las tres. En este ejemplo `True` significa bloquear.

La función siguiente reúne los pasos que ya desarrollamos. `torch.triu(..., diagonal=1)` selecciona las posiciones sobre la diagonal: las claves futuras. La tabla final tendrá filas de consultas y columnas de claves. Antes de ejecutar: ¿la última fila tendrá tres pesos iguales?

> No debería. La tercera fila tiene permitidas las tres claves por la máscara causal, pero sus puntuaciones no son iguales, pues son un producto con I que es una identidad. Las dos primeras podrían coincidir porque se basan en 0, pero el tercer valor debería ser diferente.


In [5]:
def attention(q, k, v, causal=False):
    scores_local = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if causal:
        future = torch.triu(torch.ones(scores_local.shape[-2:], dtype=torch.bool), diagonal=1)
        scores_local = scores_local.masked_fill(future, float('-inf'))
    a = torch.softmax(scores_local, dim=-1)
    return a @ v, a

q = torch.eye(3)
result, a = attention(q, q, q, causal=True)
labels = ['posición 1', 'posición 2', 'posición 3']
display(pd.DataFrame(a.numpy(), index=labels, columns=labels))
print('Sumas por consulta:', a.sum(dim=-1))
assert torch.allclose(a.sum(dim=-1), torch.ones(3), atol=1e-6, rtol=0)
assert torch.count_nonzero(torch.triu(a, diagonal=1)) == 0

,posición 1,posición 2,posición 3
posición 1,1.000000,0.000000,0.000000
posición 2,0.359542,0.640457,0.000000
posición 3,0.264458,0.264458,0.471083


Sumas por consulta: tensor([1.000000, 1.000000, 1.000000])


### Leamos lo que ocurrió

Un cero sobre la diagonal indica una exclusión, no una clave poco relevante. Las posiciones permitidas no tienen que compartir el mismo peso: sus puntuaciones pueden ser distintas. Aquí la salida coincide con los pesos porque V es la identidad; con otro V esa coincidencia dejaría de ocurrir.

Después: sigue la fila 2 desde sus puntuaciones hasta su suma. Identifica qué se comprueba con cada `assert`: si la condición no se cumple, Python detiene el bloque y pide revisar.

> Se comprueba que la suma de cada fila sea igual a 1 para que cumpla con las condiciones matemáticas requeridas para este ejercicio. En concreto, las puntuaciones de la fila 2 solo son las columnas 1 y 2, el producto da [0, 1, 0]/√3 → [0, 0.577350], tras softmax queda [0.359542, 0.640457, 0], y la suma 0.359542 + 0.640457 + 0 = 1.000000. Los dos asserts comprueban que cada fila sea una distribución válida que suma 1 y que no haya peso por encima de la diagonal (la máscara causal se respetó).

## D5 - El relleno también necesita una máscara

El **padding** es relleno usado para que varias secuencias tengan la misma longitud. No es una palabra que deba aportar contenido. Una máscara causal excluye futuro; una máscara de padding excluye relleno: responden a preguntas diferentes.

Usamos una consulta con tres puntuaciones iguales a cero. Los dos valores reales son [2,0] y [0,4]; el tercero, [0,0], es relleno. Antes de ejecutar: si el relleno vale cero, ¿puede aun así alterar el promedio? Esta es una demostración resuelta, no un ejercicio adicional de entrega.

> Sí puede alterarlo. El value del relleno es [0,0], pero igual se ve afectado por el peso. El relleno no aporta contenido, pero sí cambia cuánto pesa cada value real porque cambia el denominador del reparto."


In [6]:
padding_scores = torch.zeros(1, 3)
padding_values = torch.tensor([[2., 0.], [0., 4.], [0., 0.]])
without_padding_mask = torch.softmax(padding_scores, dim=-1)
allowed_scores = padding_scores.clone()
allowed_scores[:, 2] = float('-inf')
with_padding_mask = torch.softmax(allowed_scores, dim=-1)
print('Sin excluir relleno:', without_padding_mask @ padding_values)
print('Excluyendo relleno:', with_padding_mask @ padding_values)
print('Pesos con máscara:', with_padding_mask)
assert torch.allclose(with_padding_mask @ padding_values, torch.tensor([[1., 2.]]))

Sin excluir relleno: tensor([[0.666667, 1.333333]])
Excluyendo relleno: tensor([[1., 2.]])
Pesos con máscara: tensor([[0.500000, 0.500000, 0.000000]])


### Leamos lo que ocurrió

El relleno con valor cero también consume peso cuando no se excluye. Sin máscara dividimos la aportación entre tres opciones; con máscara, entre dos. En secuencias reales podemos combinar máscara causal y de relleno, cuidando que cada consulta válida conserve al menos una clave disponible.

## D6 - Veamos dónde vive la atención dentro del Transformer

Recibimos un lote de **dos secuencias**, con **cuatro posiciones** y **doce coordenadas** en cada posición: forma 2×4×12. `batch_first=True` pone el lote en el primer eje. Usamos números aleatorios; no son dos textos tokenizados ni un modelo entrenado.

Sumamos una señal de posición para que el contenido lleve información de orden. El vector base tiene 12 coordenadas entre 1 y 2: la posición 0 agrega ceros, la 1 agrega 0.1 veces ese vector, la 2 agrega 0.2 y la 3 agrega 0.3. Es una simplificación didáctica, no una codificación posicional completa de un modelo preentrenado.

`d_model=12` es el ancho de la representación. `nhead=3` usa tres cabezas de atención: 12/3=4 coordenadas por cabeza. Las cabezas aprenden proyecciones distintas, no les asignamos reglas lingüísticas de antemano. La red por posición, **FFN**, transforma 12→24→12 coordenadas.

Aquí usamos `TransformerEncoderLayer` con máscara causal. Es una capa de atención propia y FFN; **no es un decodificador completo** ni incluye atención cruzada. `dropout=0` evita apagar componentes al azar. `eval()` pone el bloque en modo evaluación y `no_grad()` evita calcular gradientes; ninguna de esas instrucciones entrena pesos.

Antes de ejecutar: ¿cambiará la forma de la salida o cambiarán sus valores?

>

In [7]:
# Fijamos la semilla aquí para que ejecutar otras demostraciones no altere este bloque.
torch.manual_seed(42)
block = torch.nn.TransformerEncoderLayer(
    d_model=12, nhead=3, dim_feedforward=24, dropout=0,
    batch_first=True, activation='relu', norm_first=False)
inputs = torch.randn(2, 4, 12)
position = torch.arange(4).view(1, 4, 1) / 10 * torch.linspace(1, 2, 12)
mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
block.eval()
with torch.no_grad():
    outputs = block(inputs + position, src_mask=mask)
print('Entrada:', tuple(inputs.shape), 'Posición:', tuple(position.shape),
      'Máscara:', tuple(mask.shape), 'Salida:', tuple(outputs.shape))
display(pd.DataFrame({'coordenada': np.arange(4),
    'entrada_mas_posicion': (inputs + position)[0, 1, :4].numpy(),
    'salida': outputs[0, 1, :4].numpy()}))
assert outputs.shape == inputs.shape
assert torch.isfinite(outputs).all()

Entrada: (2, 4, 12) Posición: (1, 4, 12) Máscara: (4, 4) Salida: (2, 4, 12)


,coordenada,entrada_mas_posicion,salida
0,0,-0.683242,-1.401001
1,1,-0.158692,-0.889104
2,2,1.686634,2.133542
3,3,-0.156241,-0.087871


### Leamos lo que ocurrió

La forma 2×4×12 se conserva, pero los valores cambian. La tabla muestra solo las primeras cuatro coordenadas de la posición 2 de la primera secuencia. Los índices Python comienzan en 0: `[0,1,:4]` selecciona ese fragmento.

En esta API, `src_mask=True` bloquea. No generalices esa convención a todas las funciones de PyTorch: algunas usan `True` para permitir. Consulta siempre el argumento concreto que estás usando.

## D7 - Sigamos lo que ocurre después de la mezcla

Recibimos el mismo bloque, los mismos pesos y la entrada con posición de D6. Una **conexión residual** suma la entrada a la transformación. `LayerNorm` normaliza las coordenadas de cada posición y aplica una escala y un desplazamiento aprendibles. No es softmax y no convierte la fila en probabilidades.

Con `norm_first=False`, el recorrido es: atención → suma residual → normalización → FFN → suma residual → normalización. **ReLU** cambia los números negativos por cero dentro de la FFN. Vamos a reconstruir ese recorrido para compararlo con `outputs`.

Antes de ejecutar: si repetimos las mismas operaciones con los mismos pesos, ¿qué diferencia numérica esperarías?

In [8]:
with torch.no_grad():
    z = inputs + position
    attention_part, _ = block.self_attn(z, z, z, attn_mask=mask, need_weights=False)
    after_attention = block.norm1(z + block.dropout1(attention_part))
    feed_forward = block.linear2(block.dropout(block.activation(block.linear1(after_attention))))
    rebuilt = block.norm2(after_attention + block.dropout2(feed_forward))
max_difference = (rebuilt - outputs).abs().max().item()
print('Diferencia máxima al reconstruir el bloque:', max_difference)
print('Formas después de atención y al terminar:', tuple(after_attention.shape), tuple(rebuilt.shape))
assert torch.allclose(rebuilt, outputs, atol=1e-6, rtol=0)

Diferencia máxima al reconstruir el bloque: 0.0
Formas después de atención y al terminar: (2, 4, 12) (2, 4, 12)


### Leamos lo que ocurrió

Una diferencia menor o igual a 0.000001 pasa nuestra comprobación. `1e-6` significa una millonésima; `atol` es el margen absoluto permitido. Fijamos `rtol=0` para que no se añada otro margen proporcional al valor. Esta comparación confirma el recorrido implementado bajo estas condiciones; no evalúa calidad lingüística.

## E1 - Cambiemos el futuro y observemos qué puede cambiar

Conserva el bloque de D6, sus pesos, su modo evaluación y la señal de posición. En una copia de `inputs`, suma 10 a **una coordenada de la última posición** de cada secuencia. No modifiques el original.

1. Compara las salidas de las posiciones 1–3 antes y después del cambio, con la misma máscara causal.
2. Repite las dos ejecuciones sin máscara.
3. Calcula la mayor diferencia absoluta en esas posiciones y explica qué acceso a información permite cada configuración.

No cambies pesos entre ejecuciones. Una traslación idéntica en todas las coordenadas puede quedar eliminada por la normalización; por eso cambiamos solo una. No interpretes un peso de atención como prueba de la causa de una decisión final.

**Mi predicción antes de ejecutar:** ___

**Qué conservé y qué cambié:** ___

**Resultados y conclusión, con una limitación:** ___

Este contraste apoya el paso 8 de la Actividad 3. El Laboratorio 5 conserva su alcance oficial; no añade E1 como nuevo requisito.

In [ ]:
# Tu trabajo: construye el contraste de E1 y explica sus resultados.
# Conserva inputs, position, mask y los pesos de block.

## Cierre - Tu evidencia tiene que poder explicarse

Guarda las salidas, las versiones y tus respuestas. Para el Laboratorio 5, muestra Q/K/V, dimensiones, producto, escala, softmax, mezcla y exclusión de la segunda clave; explica dónde aparece la atención dentro del Transformer. Puedes incorporar esa sección a la Actividad 3 y completar los demás pasos de su enunciado. No se añade otro informe ni se cambian sus rúbricas.

Explica con tus palabras: ¿qué recibió el cálculo?, ¿qué transformó?, ¿qué devolvió?, ¿qué demuestra la comprobación y qué no?

## Referencias para comprobar el mecanismo

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762): arquitectura original.
- [TransformerEncoderLayer, PyTorch 2.6](https://docs.pytorch.org/docs/2.6/generated/torch.nn.TransformerEncoderLayer.html): parámetros y orden de normalización.
- [Transformer y sus máscaras, PyTorch 2.6](https://docs.pytorch.org/docs/2.6/generated/torch.nn.Transformer.html): dimensiones y significado de las máscaras booleanas.

Verificado en Google Colab el 21 de octubre de 2026: demostraciones D0–D7 ejecutadas sin errores; E1 permanece pendiente para el alumno. Entorno CPU, Python 3.13.16, NumPy 2.1.3, pandas 2.2.3 y PyTorch 2.11.0+cpu. Las referencias numéricas locales usan PyTorch 2.6.0; conserva las versiones al comparar.